### TRANSFORM Address Type

### 1.create one recotd for one customer with both address.one for each address type

In [0]:
df_address = spark.read.table("gizmobox_nara.bronze.v_addresses")
display(df_address)

In [0]:
%sql
SELECT * FROM gizmobox_nara.bronze.v_addresses

In [0]:
import pyspark.sql.functions as F
df_addresses_pivot = (
    df_address
    .groupBy("customer_id")
     .pivot("address_type", ["billing", "shipping"])
     .agg(F.max("address_line_1").alias("address_line_1"),
          F.max("city").alias("city"),
          F.max("state").alias("state"),
          F.max("postcode").alias("postcode")
     )
)
display(df_addresses_pivot)

In [0]:
%sql
SELECT * 
FROM (SELECT customer_id,
        address_type,
        address_line_1,
        city,
        state,postcode
        from gizmobox_nara.bronze.v_addresses)
        PIVOT (MAX(address_line_1) AS address_line_1,
        MAX(city) AS city,
        MAX(state) AS state,
        MAX(postcode) AS postcode
        for address_type in ('shipping','billing')
        );


Write transformed data to silver schema

In [0]:
df_addresses_pivot.writeTo("gizmobox_nara.silver.py_addresses").createOrReplace()

In [0]:
df = spark.read.table("gizmobox_nara.silver.py_addresses")
display(df)

In [0]:
%sql
CREATE TABLE gizmobox_nara.silver.addresses
AS
SELECT * 
FROM (SELECT customer_id,
        address_type,
        address_line_1,
        city,
        state,postcode
        from gizmobox_nara.bronze.v_addresses)
        PIVOT (MAX(address_line_1) AS address_line_1,
        MAX(city) AS city,
        MAX(state) AS state,    
        MAX(postcode) AS postcode
        for address_type in ('shipping','billing')
        );
    
SELECT * FROM gizmobox_nara.silver.addresses